# ResumeIQ — Notebook 02: Document Parsing & Text Preprocessing

In this notebook, we explore how **ResumeIQ** ingests candidate documents (PDF and text), normalizes textual content, and extracts technical entities using word-boundary regular expressions.

## 1. Import Modules and Setup

In [ ]:
import re
from pathlib import Path
import sys

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.resume_parser import extract_pdf_data, extract_text_data
from src.skill_extractor import extract_skills, calculate_skill_match, TECH_SKILLS
print("Modules imported successfully!")

## 2. Parsing Unstructured Documents
ResumeIQ uses `pdfplumber` for multi-page PDF text extraction. Let's see how plain text extraction handles sample resume and job description text.

In [ ]:
sample_resume_text = """
Alex Johnson
Senior Machine Learning Engineer
Email: alex.johnson@example.com | GitHub: github.com/alexj

SUMMARY
Experienced ML Engineer with 4+ years designing end-to-end NLP and Computer Vision pipelines.
Proficient in Python, PyTorch, Scikit-learn, TensorFlow, SQL, Docker, and Git.
Built real-time REST APIs with FastAPI and deployed models on AWS.

EXPERIENCE
- Developed transformer-based text summarization models with Hugging Face and PyTorch.
- Automated ETL pipelines using Pandas, NumPy, and PostgreSQL.
- Containerized services using Docker and orchestrated CI/CD workflows on GitHub.

EDUCATION
B.Tech in Information Technology
"""

sample_jd_text = """
Job Title: Lead Machine Learning Engineer
Company: AI Innovators Corp

Responsibilities:
- Architect machine learning and NLP solutions for enterprise search.
- Collaborate with cross-functional teams to build REST APIs.
- Maintain scalable pipelines in Python, SQL, Docker, Kubernetes, and AWS.

Requirements:
- Strong background in Python, Machine Learning, NLP, PyTorch, and Scikit-learn.
- Experience with SQL, Git, Docker, and Kubernetes.
- Familiarity with REST APIs and cloud environments (AWS/GCP).
"""

print("Sample Resume Length:", len(sample_resume_text), "characters")
print("Sample JD Length:", len(sample_jd_text), "characters")

## 3. Skill Extraction with Word-Boundary Regular Expressions

Why use `\b` (word boundary)?
- Without boundaries, matching `"c"` would erroneously match inside `"react"`, `"scikit"`, etc.
- Matching `"sql"` might erroneously match inside `"nosql"` or other substrings.
- With `\b` + `re.escape()`, only whole words/terms are recognized accurately.

In [ ]:
resume_skills = extract_skills(sample_resume_text)
jd_skills = extract_skills(sample_jd_text)

print(f"Extracted Resume Skills ({len(resume_skills)}):")
print(resume_skills)

print(f"\nExtracted JD Skills ({len(jd_skills)}):")
print(jd_skills)

## 4. Skill Gap Diagnostics & Overlap Calculation

We calculate:
1. **Matched Skills** = $\text{Resume Skills} \cap \text{JD Skills}$
2. **Missing Skills (Gaps)** = $\text{JD Skills} \setminus \text{Resume Skills}$
3. **Skill Score** = $\frac{|\text{Matched Skills}|}{|\text{JD Skills}|} \times 100$

In [ ]:
matched = sorted(set(resume_skills).intersection(set(jd_skills)))
missing = sorted(set(jd_skills) - set(resume_skills))
skill_score = calculate_skill_match(resume_skills, jd_skills)

print(f"🎯 Skill Match Score: {skill_score:.2f}%")
print(f"\n✅ Matched Skills ({len(matched)}): {matched}")
print(f"❌ Missing Skills ({len(missing)}): {missing}")